# Reranker Fine-tuning — Ray Train
Run cells top to bottom. GPU runtime required.

In [1]:
# Cell 1 — verify GPU
import torch
print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

CUDA: True
Device: Tesla T4


In [2]:
# Cell 2 — install deps
%pip install ray[train] transformers torch qdrant-client sentence-transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 13.1 MB/s eta 0:00:00:00:0100:01


In [10]:
# Cell 3 — clone repo
!git clone https://github.com/fbarulli/RAG-a-muffin.git
%cd RAG-a-muffin

Cloning into 'RAG-a-muffin'...
remote: Enumerating objects: 3316, done.
remote: Total 3316 (delta 0), reused 0 (delta 0), pack-reused 3316 (from 1)
Receiving objects: 100% (3316/3316), 164.62 MiB | 14.87 MiB/s, done.
Resolving deltas: 100% (1044/1044), done.
/content/RAG-a-muffin


In [11]:
# Cell 4 — start Cloudflare tunnel (run on your VM before this cell)
# cloudflared tunnel --url http://localhost:6333
# paste the resulting URL below
QDRANT_URL = "https://renew-joint-trails-grown.trycloudflare.com"  

from qdrant_client import QdrantClient
client = QdrantClient(url=QDRANT_URL, prefer_grpc=False, https=True, port=443)
print(client.get_collections())

collections=[CollectionDescription(name='faqs_bge_base_en_v1_5')]


In [12]:
# Cell 5 — verify triples exist
import json, pathlib
triples_path = pathlib.Path('experiments/reranker_training/triples_sample_200.json')
triples = json.loads(triples_path.read_text())
print(f'Triples loaded: {len(triples)}')
from collections import Counter
print('Distribution:', dict(Counter(t.get('course', 'unknown') for t in triples)))

Triples loaded: 200
Distribution: {'mlops-zoomcamp': 42, 'llm-zoomcamp': 13, 'machine-learning-zoomcamp': 78, 'data-engineering-zoomcamp': 67}


In [14]:
import json, pathlib
p = pathlib.Path('configs/rerankers.json')
cfg = json.loads(p.read_text())
cfg['ray_training']['use_gpu'] = True
cfg['ray_training']['fp16'] = True
p.write_text(json.dumps(cfg, indent=2))
print('use_gpu:', cfg['ray_training']['use_gpu'])
print('fp16:   ', cfg['ray_training']['fp16'])

use_gpu: True
fp16:    True


In [ ]:

import sys
sys.path.insert(0, 'src')
import json
from pathlib import Path

# patch gpu
p = Path('configs/rerankers.json')
cfg_json = json.loads(p.read_text())
cfg_json['ray_training']['use_gpu'] = True
cfg_json['ray_training']['fp16'] = True
p.write_text(json.dumps(cfg_json, indent=2))

from rag_pipeline.ingestion.reranking.reranking_config_ray import RayTrainingConfig

cfg = RayTrainingConfig.from_rerankers_json()
triples = json.loads(Path(cfg.triples_path).read_text())
cfg_dict = cfg.to_dict()
cfg_dict['triples'] = triples

import ray
from ray.train import ScalingConfig
from ray.train.torch import TorchTrainer

ray.init(ignore_reinit_error=True, runtime_env={"working_dir": ".", "py_modules": ["src/rag_pipeline"]})

def train_loop_per_worker(config):
    import sys
    sys.path.insert(0, 'src')
    from rag_pipeline.ingestion.reranking.reranking_training_ray import train_loop_per_worker as _train
    _train(config)

trainer = TorchTrainer(
    train_loop_per_worker=train_loop_per_worker,
    train_loop_config=cfg_dict,
    scaling_config=ScalingConfig(num_workers=1, use_gpu=True),
)
result = trainer.fit()
print(result)


2026-05-23 20:03:54,633	INFO worker.py:1828 -- Calling ray.init() again after it has already been called.


(TrainController pid=15185) Requesting resources: {'GPU': 1} * 1
(TrainController pid=15185) Attempting to start training worker group of size 1 with the following resources: [{'GPU': 1}] * 1
(PlacementGroupCleaner pid=15243) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=15243) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=15243) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=15243) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=15243) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=15243) Failed to query Ray Train Con

In [ ]:
# Cell 9 — verify output
import os
out = 'experiments/reranker_models/TinyBERT-finetuned-test'
print(os.listdir(out) if os.path.exists(out) else 'NOT FOUND')